# 04 — BiLSTM with FastText embeddings

GPU strongly recommended (Colab T4 / A100 work fine).

FastText files are large (~7 GB per language). Set `FASTTEXT_PATH_<LANG>` env vars to the local `cc.<lang>.300.bin` files. If unset, embeddings start from random — model still trains, just slower convergence.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys, pathlib; sys.path.insert(0, str(pathlib.Path.cwd().parent))

from src import config as C
from src.data_utils import get_split
from src.evaluate import evaluate_and_log
from src.models import bilstm

In [ ]:
cfg = C.BiLSTMConfig()
for lang in C.LANGUAGES:
    print(f'\n=== BiLSTM :: {lang.upper()} ===')
    try:
        tr, va, te = get_split(lang)
    except FileNotFoundError as e:
        print(f'  skipped: {e}'); continue
    bundle = bilstm.train(tr, va, lang, cfg=cfg)
    y_pred = bilstm.predict(bundle, te, lang)
    metrics = evaluate_and_log(te['label'].values, y_pred, model_name='bilstm', lang=lang)
    print('  metrics:', {k: round(v, 4) for k, v in metrics.items()})
    bilstm.save(bundle, C.RESULTS_DIR / 'checkpoints' / f'bilstm_{lang}.pt')